# Programming exercise 2: Time evolution and performance of diagonalization routines

Due on Monday, 27.03.26, 20h

<font color='red'>
Common issues:
    
- Matrix multiplication, broadcasting
- complex wave functions!
- expectation values: integral vs. scalar product
- do some analytical calculations, try to understand the simplest problem instance before starting to code
- AI assistance??
</font>


## Goals of the exercise

In the first exercise we calculated the eigenenergies and eigenfunction of a quantum particle in a one-dimensional potential by representing the wave function on a discrete spatial grid and using eigh() to diagonalize the Hamiltonian matrix. This time we want to use the obtained eigenfunctions to calculate the time evolution for an arbitrary initial state and compare the performance of different routines for exact diagonalization, also exploring the benefits of sparse matrices.

In [ ]:
# load standard libraries

import numpy as np   # standard numerics library

import matplotlib.pyplot as plt   # for making plots
import matplotlib.ticker as ticker

from numpy import linalg as LA   # diagonalization and more

import time # for runtime analysis

from scipy import linalg as sLA   # diagonalization and more

from scipy.sparse.linalg import eigsh

import Comp_Quant_Dynam as cqd # our numerics module

### Exercise 1

Calculate the eigenfunctions of the harmonic oscillator on a grid as in programming exercise 1.
Calculate and plot the time evolution for an initial state which is an equal superposition between the lowest two eigenfunctions.
You can animate the plot by completing the code snippet below. Calculate the expectation value $<x>$ as a function of time. What is the functional form of the result? Does it match your expectation from the analytical solution?

Hints: Read about broadcasting rules in python. np.sum() and matrix multiplication using @ might be useful.

Solve eigenvalue problem

In [ ]:
# define the grid
L = 20
npoints = 401
xvals, dx = cqd.utility.create_xvals(L, npoints)

# build the terms of the Hamiltonian
H_pot = cqd.hamiltonians.HO_potential(xvals)
H_kin = cqd.hamiltonians.H_kinetic(xvals)

H_mat = H_pot + H_kin

# diagonalize
evals, evecs = LA.eigh(H_mat)

Time evolution of superposititions

In [ ]:
#def psit(ini_coeffs,t):
 #   return np.sum(ini_coeffs*evecs*np.exp(-1j*evals*t),axis=1)

# define the initial state
ini = np.zeros(evals.size)
ini[0] = 1 / np.sqrt(2);
ini[1] = 1 / np.sqrt(2);

# plot the initial state
plt.plot(xvals,np.abs(cqd.unitaries.t_evol_eigenbasis(ini, 0, evals, evecs))**2/dx)
plt.title("initial state")
plt.xlabel("x")
plt.ylabel("$|\\psi(x,t=0)|^2$");

Animate the solutions

In [ ]:
from matplotlib import animation
from IPython.display import HTML

In [ ]:
%%capture
fig, ax = plt.subplots()
ax.set_xlim((-L/4, L/4))
ax.set_ylim((0, 1))
ax.set_xlabel("x")
ax.set_ylabel("$|\\psi(x,t)|^2$")
line, = ax.plot([],[])

g = lambda t, init_coeffs, evals, evecs, dx: np.abs(cqd.unitaries.t_evol_eigenbasis(init_coeffs, t, evals, evecs)) ** 2 / dx # test function for plotting
args = (g, xvals, line, ini, evals, evecs, dx)

anim = animation.FuncAnimation(fig, cqd.plotting.animate,
                               frames=np.arange(0,10,0.1), # t-values
                               fargs=args, # function arguments to be passed to animate
                               interval=50, # wait time before displaying new frame in ms
                               blit=True)

In [ ]:
HTML(anim.to_jshtml())

Expectation value of x

In [ ]:
t_vals = np.arange(0, 4 * np.pi, 0.1) # time grid
x_expvals = np.zeros(t_vals.size) # allocate memory

for it, t in enumerate(t_vals):
    abs_psi = np.abs(cqd.unitaries.t_evol_eigenbasis(ini, t, evals, evecs))**2 # probability amplitude
    x_expvals[it] = xvals @ abs_psi # x expectation value

# analytical solution
amplitude = np.conjugate(evecs[:,0]) @ (xvals*evecs[:,1]) # matrix element
sol_analyt = amplitude*np.cos(t_vals)
    
plt.plot(t_vals, x_expvals, label = "numerical solution")
plt.plot(t_vals, sol_analyt,'k--', label = "analytical solution")
plt.xlabel("t")
plt.ylabel("$<x>$")
plt.legend(loc = "lower center")
plt.show()

The analytical solution is just an oscillation of the mean with frequency 1 ($\omega=1$), as for a classical particle. The amplitude depends on the overlap matrix element between the two components of the superposition.

### Exercise 2

Calculate the time evolution for arbitrary intial states by decomposing them into a superpostion of eigenstates. Test your implementation for a coherent state, i.e. a Gaussian state that is shifted by $x_0$:
$$
\psi_0(x) = \frac{1}{\pi^{1/4}}e^{-(x-x_0)^2/2}
$$
Again, animate the resulting dynamics and also calculate the expectation value of x as a function of time. Describe and interpret your observations.

Optional: Play with other initial conditions and other potentials. For example take the double well potential from problem sheet 1 and initialize the system in a state localized on one side. What happens? How do the dynamics depend on the barrier hieght, i.e. on $\lambda$? Also instructive: Build up the coherent state by taking into account more and more of the eigenstates in the initial superposition and observe what happens to the time evolution.

In [ ]:
# define the initial state
x0 = -1
psi0 = np.exp(-(xvals - x0) ** 2 / 2) / np.sqrt(np.sqrt(np.pi)) * np.sqrt(dx)
# normalize s.t. vector norm = 1, not integral norm
ini_coh = cqd.unitaries.init_coeffs_eigenbasis(psi0, evecs)

LA.norm(ini)

In [ ]:
%%capture
fig, ax = plt.subplots()
ax.set_xlim((-L/4, L/4))
ax.set_ylim((0, 1))
ax.set_xlabel("x")
ax.set_ylabel("$|\\psi(x,t)|^2$")
line, = ax.plot([],[])

args = (g, xvals, line, ini_coh, evals, evecs, dx)


anim = animation.FuncAnimation(fig, cqd.plotting.animate,
                               frames=np.arange(0,10,0.1), # t-values
                               fargs=args, # function arguments to be passed to animate
                               interval=50, # wait time before displaying new frame in ms
                               blit=True)

In [ ]:
HTML(anim.to_jshtml())

Expectation value of x

In [ ]:
x_expvals_coh = np.zeros(t_vals.size) # allocate memory
for it, t in enumerate(t_vals):
    abs_psi = np.abs(cqd.unitaries.t_evol_eigenbasis(ini_coh, t, evals, evecs))**2 # probability amplitude
    x_expvals_coh[it] = xvals @ abs_psi # x expectation value

# analytical solution
sol_analyt_coh = x0*np.cos(t_vals)
    
plt.plot(t_vals, x_expvals_coh, label = "numerical solution")
plt.plot(t_vals, sol_analyt_coh,'k--', label = "analytical solution")
plt.xlabel("t")
plt.ylabel("$<x>$")
plt.legend(loc = "upper center")
plt.show()

This again oscillated sinusoidaly like a classical particle. This time the shape (width) of the probability distribution is conserved. Coherent states evolve without dispersion.

### Exercise 3

How does the runtime scale with the basis size (number of grid points)? Increase the number of gridpoints from 101 to 1601 in steps of 100 and record the time needed for the exact diagonalization. Plot the runtime as a function of basis size on a double logarithmic scale. Also try out the routine scipy.linalg.eigh_tridiagonal() and compare its runtime to np.eigh().

The purpose of this exercise is to sensibilize you that using routines that take into account as much information about the problem to be solved can be a huge advantage in terms of computation time.
The outcome may vary depending on the machine you are using. You are encouraged to explore the documentation of the respective functions and find out what kind of routines they are using underneath.

Hints: You can e.g. use the "time" module you can measure the time it takes to do the diagonalization.

In [ ]:
#n_vals = np.arange(1001, 2801, 100)
n_vals = np.arange(101, 801, 100)

n_reps = 10

runtimes = np.zeros((n_vals.size, n_reps))
runtimes_tri = np.zeros((n_vals.size, n_reps))

for n_idx, nn in enumerate(n_vals):
    xvals_scaling, dx_scaling = cqd.utility.create_xvals(L, nn)
            
    # terms of the Hamiltonian
    H_pot = cqd.hamiltonians.HO_potential(xvals_scaling)
    H_kin = cqd.hamiltonians.H_kinetic(xvals_scaling)

    H_mat = H_pot + H_kin

    d = np.diagonal(H_mat)
    e = np.diagonal(H_mat, offset=1)
    
    for rep_idx in range(n_reps):

        ti = time.perf_counter()
        evals, evecs = LA.eigh(H_mat)
        tf = time.perf_counter()
        runtimes[n_idx, rep_idx] = tf - ti

        ti = time.perf_counter()
        evals, evecs = sLA.eigh_tridiagonal(d, e)
        tf = time.perf_counter()
        runtimes_tri[n_idx, rep_idx] = tf - ti

runtimes_avg = np.mean(runtimes, axis=1)
runtimes_tri_avg = np.mean(runtimes_tri, axis=1)

print(runtimes_avg)

In [ ]:
cmap = plt.get_cmap("tab10")
plt.loglog(n_vals, runtimes_avg, label = "Runtimes eigh")
plt.loglog(n_vals, runtimes_tri_avg, label = "Runtimes eigh_tridiagonal")
 
plt.loglog(
    n_vals, (0.525 * 10 ** (-3) * n_vals) ** 3, color = cmap(0), ls = "--",
    label = "Runtime expectation eigh"
)
plt.loglog(n_vals, (0.3 * 10 ** (-3) * n_vals) ** 2, color = cmap(1), ls = "--",
    label = "Runtime expectation eigh_tridiagonal"
)
plt.xlabel("gridpoints")
plt.ylabel("runtime")

plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(500))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter("%d"))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(100))
plt.gca().xaxis.set_minor_formatter(ticker.NullFormatter())
plt.legend(loc = "upper left")
plt.show()

Interpretation: Quadratic scaling with n_points. In general one would expect ^3 for exact diagonalization, but tri-diagonal matrices can be solved by a simple recursion. eigh_tridiagonal exploits this optimally, while eigh scales a bit worse.

### Exercise 4

Since our Hamiltonian matrix contains a lot of zeros it makes sense to use sparse matrices. Hamiltonian matrices of physical systems are very often sparse. If we are only interested in the low lying eigenstates, the Lanczos algorithm provides a very efficient way of obtaining those. Calculate the lowest (e.g. 20) eigenvalues, using sparse matrices. Compare the runtime to your previous implementations. For diagonalising a sparse symmetric matrix, you can use scipy.sparse.linalg.eigsh(...)

Does the observed scaling of the runtime match your expectations? Again the outcome may depend on the hardware you are using.

Hints: For building the Hamiltonian as a sparse matrix, the function scipy.sparse.diags(...) might be helpful. Think about what the "which" option of scipy.sparse.linalg.eigsh(...) should be set to.

Sparse matrix implementation

In [ ]:
npoints = 401
xvals, dx = cqd.utility.create_xvals(L, npoints)

# terms of the Hamiltonian
H_pot_sparse = cqd.hamiltonians.HO_potential_sparse(xvals)
H_kin_sparse = cqd.hamiltonians.H_kinetic_sparse(xvals)

H_sparse = H_kin_sparse + H_pot_sparse
#print(Hsparse.toarray())

evals_sparse, evecs_sparse = eigsh(H_sparse, k=20, which='SA')

Check the results

In [ ]:
print(evals_sparse)
plt.plot(xvals,np.abs(evecs_sparse[:, 0]) ** 2 / dx)
plt.xlabel("x")
plt.ylabel("$|\\phi_n(x)|^2$");

Runtime analysis

In [ ]:
runtimes_sparse = np.zeros((n_vals.size, n_reps))

for n_idx, nn in enumerate(n_vals):
    xvals_scaling, dx_scaling = cqd.utility.create_xvals(L, nn)
    # terms of the Hamiltonian
    H_pot_sparse = cqd.hamiltonians.HO_potential_sparse(xvals_scaling)
    H_kin_sparse = cqd.hamiltonians.H_kinetic_sparse(xvals_scaling)
    H_sparse = H_kin_sparse + H_pot_sparse
    for rep_idx in range(n_reps):

        ti = time.perf_counter()
        e, v = eigsh(H_sparse, k=2, which='SA')
        tf = time.perf_counter()
        runtimes_sparse[n_idx, rep_idx] = tf - ti

runtimes_sparse_avg = np.mean(runtimes_sparse, axis=1)
print(runtimes_sparse_avg)

In [ ]:
plt.loglog(n_vals, runtimes_avg, label = "Runtimes eigh")
plt.loglog(n_vals, runtimes_tri_avg, label = "Runtimes eigh_tridiagonal")
plt.loglog(n_vals, runtimes_sparse_avg, label = "Runtimes eigsh")

plt.loglog(
    n_vals, (0.525 * 10 ** (-3) * n_vals) ** 3, color = cmap(0), ls = "--",
    label = "Runtime expectation eigh"
)
plt.loglog(n_vals, (0.3 * 10 ** (-3) * n_vals) ** 2, color = cmap(1), ls = "--",
    label = "Runtime expectation eigh_tridiagonal"
)
#plt.loglog(n_vals, (0.2 * 10 ** (-3) * n_vals) ** 1, color = cmap(2), ls = "--",
#    label = "Runtime expectation eigh_tridiagonal"
#)
plt.xlabel("gridpoints")
plt.ylabel("runtime")
plt.legend(loc = "upper left")
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(500))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter("%d"))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(100))
plt.gca().xaxis.set_minor_formatter(ticker.NullFormatter())
plt.show()

For sparse matrix method, scaling should be close to linear. An advantage compared to eigh_tridiagonal() is not yet visible for the basis sizes considered.